# 7 · Discursos e apresentações — quem falou o quê (com nome e data)

**Objetivo:** complementar o índice das atas (notebook 06) com o corpus onde
**a posição é assinada**: discursos e apresentações de diretores e do
presidente do BCB, cada trecho rastreável até **autor, data e título**.

**Como o corpus foi obtido:** o portal `bcb.gov.br` é uma SPA sem listagem em
HTML; a API pública interna foi recuperada por engenharia reversa dos bundles
JS (`/api/servico/sitebcb/discursos?ano=YYYY`). O índice completo tem
**1.556 discursos (2002–2026)** (`dados/processado/discursos_indice.csv`).

**Recorte "foco" — uma ESCOLHA documentada (não é o corpus inteiro):**
- ano ≥ 2021 (mesma janela das 47 atas do Copom);
- tema monetário no título (Copom, Selic, juros, inflação, política monetária,
  cenários econômicos, expectativas, Relatório de Política Monetária);
- autoridade de decisão (Presidente/Diretor) ou divulgação de Relatório.
→ **46 discursos** com PDF e texto extraído.

**O que este índice NÃO é:**
- **não é** classificação de intenção: achar a frase "juro real" num discurso
  não significa que o autor "defende o patamar" — o trecho é material para
  leitura humana;
- **não é** NLP: as mesmas **regexes das atas** (`ac.TEMAS`) localizam
  frases; a régua é idêntica para permitir comparação direta;
- **não extrapola**: onde o texto não fala, o índice fica vazio.

**Perguntas do projeto:** P1 e P5 — aqui ficam os argumentos *assinados* que
sustentaram o patamar já quantificado nos notebooks 00–05.


In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA
from src.processa import discursos as dsp
from src.processa import atas_copom as ac

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
cat = dsp.carrega_catalogo()
print(f"foco: {len(cat)} discursos ({cat.ano.min()}–{cat.ano.max()}), "
      f"{cat.autor.nunique()} autoridades")
print("\ndiscursos por autor:")
print(cat.autor.value_counts().to_string())
print("\npor ano:")
print(cat.groupby("ano").size().to_string())
cat[["id", "data", "autor", "titulo"]].head(6)

foco: 46 discursos (2021–2026), 8 autoridades

discursos por autor:
autor
Diogo Guillen            16
Bruno Serra Fernandes    10
Paulo Picchetti           6
Nilton David              5
Fabio Kanczuk             4
Roberto Campos Neto       2
Fernanda Guardado         2
Paulo Souza               1

por ano:
ano
2021    11
2022     8
2023     8
2024     5
2025     9
2026     5


,id,data,autor,titulo
0,1262,2021-03-04,Bruno Serra Fernandes,"Discurso do Diretor de Política Monetária, Bru..."
1,1266,2021-03-25,Fabio Kanczuk,"Apresentação do Diretor de Política Econômica,..."
2,1268,2021-04-06,Roberto Campos Neto,"Apresentação do Presidente do Banco Central, R..."
3,1271,2021-04-08,Bruno Serra Fernandes,"Apresentação do Diretor de Política Monetária,..."
4,1285,2021-05-14,Bruno Serra Fernandes,"Apresentação do Diretor de Política Monetária,..."
5,1301,2021-06-24,Fabio Kanczuk,"Apresentação do Diretor de Política Econômica,..."


In [3]:
mat = dsp.temas()
temas_ord = [t for t in ac.TEMAS if t in mat.columns]
matriz = mat.groupby("ano")[temas_ord].sum()

fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(matriz.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(temas_ord)), [t.replace(" ", "\n") for t in temas_ord],
              fontsize=9)
ax.set_yticks(range(len(matriz.index)), matriz.index)
for i in range(len(matriz.index)):
    for j in range(len(temas_ord)):
        v = matriz.values[i, j]
        if v:
            ax.text(j, i, int(v), ha="center", va="center", fontsize=9,
                    color="black" if v < matriz.values.max() * 0.6 else "white")
ax.set_title("Discursos 2021–2026 (recorte foco): nº de frases por ano e tema")
fig.colorbar(im, ax=ax, shrink=0.8, label="frases")
salvar(fig, "07_temas_por_ano")

fig -> viz/figs/07_temas_por_ano.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/07_temas_por_ano.png')

In [4]:
a = dsp.por_autor()
cols = [t for t in ac.TEMAS if t in a.columns]
autor_mat = a.set_index("autor")[cols]

fig, ax = plt.subplots(figsize=(11, 3.8))
im = ax.imshow(autor_mat.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(cols)), [t.replace(" ", "\n") for t in cols], fontsize=9)
ax.set_yticks(range(len(autor_mat.index)), autor_mat.index, fontsize=9)
for i in range(len(autor_mat.index)):
    for j in range(len(cols)):
        v = autor_mat.values[i, j]
        if v:
            ax.text(j, i, int(v), ha="center", va="center", fontsize=8.5,
                    color="black" if v < autor_mat.values.max() * 0.55 else "white")
ax.set_title("Recorte foco: frases com cada tema, por autoridade (inventário)")
fig.colorbar(im, ax=ax, shrink=0.8, label="frases")
salvar(fig, "07_autores_temas")

fig -> viz/figs/07_autores_temas.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/07_autores_temas.png')

In [5]:
# mesma régua (ac.TEMAS) nos dois corpora: participação de cada tema
# no total de frases indexadas de cada corpus
mat_a = ac.temas()
cols = [t for t in ac.TEMAS if t in mat.columns and t in mat_a.columns]
fd = mat[cols].sum() / mat[cols].sum().sum()
fa = mat_a[cols].sum() / mat_a[cols].sum().sum()
cmp = pd.DataFrame({"discursos": fd * 100, "atas": fa * 100})
cmp["dif"] = cmp.discursos - cmp.atas
cmp = cmp.sort_values("dif")

fig, ax = plt.subplots(figsize=(10, 4.6))
y = np.arange(len(cmp))
ax.barh(y - 0.2, cmp.atas, height=0.4, color=AZUL, label="atas do Copom (46)")
ax.barh(y + 0.2, cmp.discursos, height=0.4, color=LARANJA,
        label="discursos (46, recorte foco)")
ax.set_yticks(y, cmp.index, fontsize=9)
ax.set_xlabel("% das frases indexadas do corpus")
ax.set_title("Mesma régua, dois corpora: onde cada um concentra o argumento")
ax.legend(frameon=False, fontsize=9)
for i, (v, w) in enumerate(zip(cmp.atas, cmp.discursos)):
    ax.text(v + 0.4, i - 0.2, f"{v:.1f}", va="center", fontsize=8, color=AZUL)
    ax.text(w + 0.4, i + 0.2, f"{w:.1f}", va="center", fontsize=8,
            color=VERMELHO)
salvar(fig, "07_discursos_vs_atas")
print(cmp.round(1).to_string())

fig -> viz/figs/07_discursos_vs_atas.png
                          discursos  atas   dif
tema                                           
risco fiscal                   2.40 10.60 -8.20
credibilidade                  2.00  9.90 -7.90
meta de inflação              11.10 18.90 -7.80
histórico de desencontro       0.70  1.70 -1.10
câmbio                        18.80 19.40 -0.60
juro real ex ante              1.00  0.10  0.90
surpresa inflacionária         4.10  1.10  3.00
juro real                      4.50  0.80  3.70
mercado financeiro            23.20 17.00  6.20
risco externo                 32.40 20.60 11.80


In [6]:
# inventário citável: cada trecho com autor + data + id do discurso
df_trechos = pd.read_parquet(p / "dados" / "processado" / "discursos_trechos.parquet")
print(f"trechos indexados: {len(df_trechos):,} em {df_trechos.id.nunique()} "
      f"discursos, {df_trechos.tema.nunique()} temas")
print(df_trechos.tema.value_counts().to_string())

for tema in ["juro real ex ante", "credibilidade"]:
    sub = df_trechos[df_trechos.tema == tema].sort_values("data")
    print("\n" + "=" * 100)
    print(f"TEMA: {tema}  ({len(sub)} trechos)")
    for _, r in sub.iterrows():
        print(f"\n-- {r.data} · {r.autor} · id {r.id}")
        print(f"   {r.trecho[:420]}")

trechos indexados: 911 em 45 discursos, 10 temas
tema
risco externo               295
mercado financeiro          211
câmbio                      171
meta de inflação            101
juro real                    41
surpresa inflacionária       37
risco fiscal                 22
credibilidade                18
juro real ex ante             9
histórico de desencontro      6

TEMA: juro real ex ante  (9 trechos)

-- 2021-12-16 · Fabio Kanczuk · id 1381
   • Exercício mostra o descasamento temporal entre os efeitos na inflação de choques de custos, no caso preço de commodities, e os da reação de política monetária. Taxa Selic real ex-ante e taxa de juros real neutra estimada % a. Resposta da inflação a choques de commodities e de política monetária Acumulada em quatro trimestres (p.

-- 2021-12-16 · Fabio Kanczuk · id 1381
   na Selic por 4T * Elevação de 2 p. na Selic por 4T * *Resposta em valores absolutos Trimestres -2 0 2 4 6 8 10 12 14 IV 2003 II 2005 IV 2006 II 2008 IV 2009 II 2011 IV

## O que o índice mostra

1. **Aqui sim há nome.** O foco tem **8 autoridades** (Diogo Guillen, Bruno
   Serra, Paulo Picchetti, Nilton David, Fabio Kanczuk, Roberto Campos Neto,
   Fernanda Guardado, Paulo Souza) — diferente das atas, que registram o Comitê
   sem votos nominais. Mas *falar de* um tema num discurso **não é** votar por
   ele: o documento é uma posição pública assinada, não um registro de decisão.

2. **Dois corpora, dois públicos** (mesma régua, figura `07_discursos_vs_atas`):
   - **discursos** concentram-se em *risco externo* (32% vs 21%) e *mercado
     financeiro* (23% vs 17%) — apresentações para plateia externa (RPM, FGV,
     seminários) descrevem o cenário global;
   - **atas** concentram-se em *risco fiscal* (11% vs 2%) e *credibilidade*
     (10% vs 2%) — o documento de decisão discute ancoragem e sustentabilidade;
   - **juro real ex ante** aparece **~10× mais** nos discursos (1,0% vs 0,1%):
     a discussão de prêmio real sobre a neutra (o "Taylor +6 p.p." do notebook
     05) **ocorre nas apresentações, não nas atas**.

3. **Silêncio consistente:** "swap" = 0 ocorrências **também** nos 46
   discursos — o derivativo cambial não é citado em nenhum dos dois corpora
   oficiais (mesmo resultado do notebook 06).

4. **Onde falam de juro real ex ante:** Fabio Kanczuk (RTI 4T2021, compara
   "Selic real ex-ante e taxa de juros real neutra estimada"), Paulo Picchetti
   (7/6/2024: "Taxa de juros real também tem diminuído no Brasil, mais do que
   nos pares") e Diogo Guillen (boxe do RPM sobre Focus e prêmio a termo) —
   trechos com autor, data e PDF para citação direta.

**Reprodutibilidade:**
`PYTHONPATH=. python -m src.extratores.discursos` (índice 1.556) →
`... discursos --foco` (46 PDFs + textos) →
`PYTHONPATH=. python -m src.processa.discursos` (parquet
`dados/processado/discursos_trechos.parquet`, 911 trechos).

**Limitações:** slides de divulgação geram ruído (números de gráfico viram
"frases" — por isso a régua exige ≥4 palavras e a leitura é humana); o
recorte foco é uma escolha — ampliá-lo muda os totais (a comparação
normalizada por participação é mais robusta que contagens brutas); 1 discurso
do foco (id 1705) não contém nenhum tema do índice. O corpus complementar
restante — **manifestos de entidades do mercado** — ainda não foi coletado.